In [6]:
import os, sys

# Resolve code/beh_ephys_analysis (the folder containing `utils`) relative to this
# file's location, so imports work no matter where the repo is checked out.
_anchor = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.path.abspath(os.getcwd())
_found = False

while _anchor != os.path.dirname(_anchor):
    _beh_ephys_root = os.path.join(_anchor, "code", "beh_ephys_analysis")
    if os.path.isdir(_beh_ephys_root) and os.path.isdir(os.path.join(_beh_ephys_root, "utils")):
        if _beh_ephys_root in sys.path:
            sys.path.remove(_beh_ephys_root)
        sys.path.insert(0, _beh_ephys_root)
        _found = True
        break
    _anchor = os.path.dirname(_anchor)

# Fallback: try absolute path if we're in a Capsule environment
if not _found:
    _fallback = "/root/capsule/code/beh_ephys_analysis"
    if os.path.isdir(_fallback) and os.path.isdir(os.path.join(_fallback, "utils")):
        if _fallback in sys.path:
            sys.path.remove(_fallback)
        sys.path.insert(0, _fallback)
        _found = True

if not _found:
    raise ImportError(f"Could not locate code/beh_ephys_analysis/utils from {os.getcwd()}")

from utils.capsule_migration import CAPSULE_ROOT, capsule_directories
from utils.pupil_utils import load_pupil
import pandas as pd
import numpy as np
from utils.beh_functions import parseSessionID


# load pupil

In [2]:
session = 'behavior_ZS062_2021-05-03_13-58-20'
pupil_data = load_pupil(session) # returns none is pupil data is not available for the session

Loading pupil file: mZS062d20210503_pupil.mat


In [3]:
timestamps = pupil_data['pupil_times'] # unit = second
pupil_diameter = pupil_data['pupil_diameter']

# load lick table (from video)

In [11]:
session = 'behavior_791691_2025-06-27_13-54-30'
video_data_file = str(capsule_directories()['data_dir']) + '/all_tongue_movements_04022026/all_tongue_movements_04022026.parquet'
all_lick_df = pd.read_parquet(video_data_file)
session_video_list = all_lick_df['session'].unique().tolist()
# load data
animal_id, session_time_curr, _  = parseSessionID(session)
# find the corresponding session in the session list withs same animalsl_id and closest session time
session_list_curr = [s for s in session_video_list if str(s).startswith(f'behavior_{animal_id}')]
if len(session_list_curr) == 0:
    print(f"No session found for {session}. Skipping.")
else:
    session_index = np.argmin([abs((parseSessionID(s)[1] - session_time_curr).total_seconds()) for s in session_list_curr])
    if np.min([abs((parseSessionID(s)[1] - session_time_curr).total_seconds()) for s in session_list_curr])>60:
        print(f"No closely matched session found for {session}. Skipping.")
    else:
        session_video = session_list_curr[session_index]
        session_licks = all_lick_df[all_lick_df['session'] == session_video].copy().reset_index(drop=True)
        print(f"Found matched session {session_video} for {session}. Loaded {len(session_licks)} licks.")

Found matched session behavior_791691_2025-06-27_13-54-27 for behavior_791691_2025-06-27_13-54-30. Loaded 9840 licks.


In [12]:
print(session_licks.head())

   movement_id  start_time    end_time  duration       min_x       max_x  \
0            1 -373.240768 -373.172768  0.068000  325.110830  347.039850   
1            2 -373.104768 -373.030752  0.074016  300.085807  344.293261   
2            3 -372.926752 -372.852768  0.073984  283.389324  332.721128   
3            4 -372.744768 -372.674752  0.070016  322.812046  353.050802   
4            5 -371.698784 -371.618784  0.080000  301.979429  329.658644   

        min_y       max_y       min_xv       max_xv  ...  \
0  211.408207  254.050388 -3508.686129   486.874648  ...   
1  219.637226  255.748344 -2039.116933  2403.376352  ...   
2  217.824701  257.794598 -2354.915576  1999.775496  ...   
3  227.219074  262.811447 -2244.435716  1289.044759  ...   
4  224.192319  280.041273 -1462.934925  1554.890409  ...   

   movement_number_in_trial  cue_response_movement_number  \
0                      <NA>                          <NA>   
1                      <NA>                          <NA>   